# Component 2 — Sustained Vowel /a/ Baseline ("AH" dataset)

Dataset: `data/vowel_task/` — sustained /a/ recordings, PD ("PwPD") vs HC
(80 recordings after dropping 1 unvoiced file: 40 PD / 40 HC).

This is the closer match to Component 2's actual intended design ("aaaaaa")
compared to the `ReadText` stand-in baseline in
`component2_phonation_baseline.ipynb`. Same feature extractor
(`src/component2_phonation/features.py`), same training/eval code
(`src/component2_phonation/evaluation.py`) — only the dataset loader differs.

**Caveat:** PD (mean age 67.0) and HC (mean age 47.6) groups are not
age-matched here — keep that in mind when interpreting accuracy.

In [ ]:
import sys, os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import pandas as pd
import matplotlib.pyplot as plt

from src.component2_phonation.vowel_dataset import build_vowel_features, ensure_extracted
from src.component2_phonation.features import FEATURE_NAMES

RESULTS_DIR = os.path.join(REPO_ROOT, "results", "component2_phonation")
FEATURES_CSV = os.path.join(RESULTS_DIR, "vowel_a_features.csv")

In [ ]:
if os.path.exists(FEATURES_CSV):
    df = pd.read_csv(FEATURES_CSV)
    print(f"Loaded cached features from {FEATURES_CSV}")
else:
    ensure_extracted()
    df = build_vowel_features()
    os.makedirs(RESULTS_DIR, exist_ok=True)
    df.to_csv(FEATURES_CSV, index=False)

print("Shape:", df.shape)
print(df["label_name"].value_counts())
df.head()

## Exploratory analysis

In [ ]:
print("Age by group:")
display(df.groupby("label_name")["age"].describe())

summary = df.groupby("label_name")[FEATURE_NAMES].mean().T
display(summary)

In [ ]:
plt.figure(figsize=(7, 5))
plt.boxplot(
    [df[df["label_name"] == "HC"]["jitter"], df[df["label_name"] == "PD"]["jitter"]],
    tick_labels=["HC", "PD"],
)
plt.ylabel("Jitter")
plt.title("Jitter: PD vs HC (sustained /a/)")
plt.show()

## Train + evaluate

Re-running this refreshes `models/component2_phonation_vowel_rf.joblib` and
`results/component2_phonation/vowel_*`.

In [ ]:
from src.component2_phonation import train_vowel as train_vowel_module
train_vowel_module.main()

## Predict on a single recording

In [ ]:
from src.component2_phonation.predict import predict

sample_path = os.path.join(REPO_ROOT, "data", "vowel_task", "raw", "PD_AH", "AH_545616858-3A749CBC-3FEB-4D35-820E-E45C3E5B9B6A.wav")
result = predict(sample_path, task="vowel")
print(result["audio_path"])
print("Prediction:", result["prediction"])
print(f"P(HC) = {result['probability_hc']:.4f}   P(PD) = {result['probability_pd']:.4f}")